#### pandasなどのimport、解析前に環境設定

In [31]:
import pandas as pd
from Bio import SeqIO, Entrez
import matplotlib as mpl
from matplotlib.font_manager import FontProperties
mpl.rcParams['svg.fonttype'] = 'none'
# Path to your Arial.ttf file
arial_font_path = '/Users/izumikida/fonts/Arial.ttf'
# Create a FontProperties object for the Arial font
arial_font = FontProperties(fname=arial_font_path)
mpl.rcParams['font.family'] = arial_font.get_name()
# import matplotlib.pyplot as plt
# import seaborn as sns
import numpy as np
import time
import subprocess

In [32]:
def fasta_to_dataframe(fasta_file):
    ids = []
    sequences = []
    for record in SeqIO.parse(fasta_file, "fasta"):
        ids.append(record.id)
        sequences.append(str(record.seq))

    df = pd.DataFrame({
        "ID": ids,
        "seq": sequences
    })
    return df

file名随時変更。AliViewでposition確認。

In [33]:
# ORFのpositionを定義する関数 parse_positions
def parse_positions(pos_file):
    ORFs = {}
    with open(pos_file, 'r') as f:
        for line in f:
            line = line.strip()
            if not line:  # skip empty line
                continue
            parts = line.split(',')
            # parts[0] = ORF name, parts[1] = start pos, parts[2] = end pos
            orfname = parts[0]
            start = int(parts[1])
            end = int(parts[2])
            ORFs[orfname] = [start, end]
    return ORFs
FCoV_pos_path = '/Users/izumikida/FCoV-evolution/FCoV2_WSU791146_ORF_pos.csv'
FCoV_pos = parse_positions(FCoV_pos_path)
FCoV_pos

{'Orf1ab': [311, 20439],
 'S': [20436, 24794],
 'Orf3a': [24861, 25076],
 'Orf3b': [25021, 25236],
 'Orf3c': [25233, 25967],
 'E': [25954, 26202],
 'M': [26213, 27001],
 'N': [27014, 28147],
 'Orf7a': [28152, 28457],
 'Orf7b': [28462, 29082]}

In [34]:
#get the alignment positions based on one ref's pos
def al_pos(positions, al_file, ref):

    records = SeqIO.to_dict(SeqIO.parse(al_file, 'fasta'))
    numgaps = [str(records[n].seq).count('-') for n in list(records)]
    if len(ref) == 0:
        ref = list(records)[numgaps.index(min(numgaps))]
    refseqone = str(records[ref].seq)

    map = {}
    c = 0
    for i in range(len(refseqone)):
        if str(c) not in list(map):
            map.update({str(c):[i]})
        else:
            map[str(c)].append(i)
        if refseqone[i] != '-':
            c = c +1

    finalpos = [max(map[str(x)]) for x in positions]
    
    return finalpos


ORFs = {}
for name, coords in FCoV_pos.items():
    ORFs[name] = al_pos(coords, 'FCoV2_modified_250903_linsi.fas', 'NC_002306.3')
print(ORFs)

{'Orf1ab': [314, 20448], 'S': [20445, 24827], 'Orf3a': [24913, 25128], 'Orf3b': [25073, 25295], 'Orf3c': [25292, 26027], 'E': [26014, 26266], 'M': [26277, 27074], 'N': [27087, 28221], 'Orf7a': [28226, 28531], 'Orf7b': [28536, 29156]}


ORF毎にaliginment後のpositionで切り分ける　ATGで始まりが揃っているかと終わりが終始コドンで揃っているか確認

In [35]:
# エラーしてchatgpt参考に追加したスクリプト
from Bio.Seq import Seq

In [36]:
def extract_ORFs(orfd, wg_al_file, pr_name, ungap = 'Y', translate = 'Y'):

    recdict = SeqIO.to_dict(SeqIO.parse(wg_al_file, 'fasta'))

    for orf in list(orfd):
        
        # refseq = str(recdict[ref].seq)
        b = orfd[orf]
        start = b[0]-1
        end = b[1]
        # tp = 0
        # alp = 0
        # alstart = 0
        # alend = 0
        # for n in refseq:
                # if tp == start:
                        # alstart = alp
                # if tp == end:
                        # alend = alp
                # alp = alp+1
                # if n != '-':
                        # tp = tp+1
        
        alfile = 'al_%s_%s_nt_prelim.fas'%(pr_name, orf)
        ungapfile = '%s_%s_nt.fas'%(pr_name,orf)
        aafile = '%s_%s_aa.fas'%(pr_name, orf)
        
        fasta = ''
        ungapfasta = ''
        aafasta = ''    
        
        for virname in list(recdict):            
            # thisseq = str(recdict[virname].seq)[alstart:alend]
            thisseq = str(recdict[virname].seq)[start:end]
            fasta = fasta + '>%s\n%s\n'%(virname, thisseq)
            
            if ungap == 'Y':
                ungapseq = thisseq.replace('-', '')                
                ungapfasta = ungapfasta + '>%s\n%s\n'%(virname, ungapseq)
                
            if translate == 'Y':
                aaseq = Seq(thisseq.replace('-', '')).translate()         
                aafasta = aafasta + '>%s\n%s\n'%(virname, str(aaseq))
                    
        with open(alfile, 'w+') as out:
            out.write(fasta[:-1])            
        with open(ungapfile, 'w+') as out:
            out.write(ungapfasta[:-1])            
        with open(aafile, 'w+') as out:
            out.write(aafasta[:-1])            
            

prelimal = '/Users/izumikida/FCoV-evolution/FCoV_wg/250903/FCoV2_modified_250903_linsi.fas'
project = 'FCoV2_comp'
extract_ORFs(ORFs, prelimal, project, 'Y', 'Y')

/opt/anaconda3/envs/FCoV/lib/python3.13/site-packages/Bio/Seq.py:2879: BiopythonWarning: Partial codon, len(sequence) not a multiple of three. Explicitly trim the sequence or add trailing N before translation. This may become an error in future.
  warnings.warn(


ExtractしたORFをalignment　AAのfileと読み枠を考慮して塩基配列に戻したfileを作成

In [12]:
def ORFal_aa_to_nt(orfd, pr_name, mafft_path, p2n_path):
    
    for orf in list(orfd):
        
        aafile = '%s_%s_aa.fas'%(pr_name, orf)
        alaafile = 'al_%s_%s_aa.fas'%(pr_name, orf)
        ntoutfile = 'al_%s_%s_nt.fas'%(pr_name, orf)
        ntfile = '%s_%s_nt.fas'%(pr_name, orf)


        cmd = '%s --localpair --maxiterate 1000 %s > %s'%(mafft_path, aafile, alaafile)
        subprocess.Popen([cmd], shell = True,close_fds=True).wait()
        
        cmd = 'perl %s %s %s -output fasta > %s'%(p2n_path, alaafile, ntfile, ntoutfile)
        subprocess.Popen([cmd], shell = True,close_fds=True).wait()

mafft_path = 'mafft'
p2n_path = '/Applications/pal2nal_v14/pal2nal.pl'

ORFal_aa_to_nt(ORFs, project, mafft_path, p2n_path)

outputhat23=16
treein = 0
compacttree = 0
stacksize: 8176 kb
rescale = 1
All-to-all alignment.
tbfast-pair (aa) Version 7.526
alg=L, model=BLOSUM62, 2.00, -0.10, +0.10, noshift, amax=0.0
0 thread(s)

outputhat23=16
Loading 'hat3.seed' ... 
done.
Writing hat3 for iterative refinement
rescale = 1
Gap Penalty = -1.53, +0.00, +0.00
tbutree = 1, compacttree = 0
Constructing a UPGMA tree ... 
    0 / 9
done.

Progressive alignment ... 
STEP     8 /8 
done.
tbfast (aa) Version 7.526
alg=A, model=BLOSUM62, 1.53, -0.00, -0.00, noshift, amax=0.0
1 thread(s)

minimumweight = 0.000010
autosubalignment = 0.000000
nthread = 0
randomseed = 0
blosum 62 / kimura 200
poffset = 0
niter = 16
sueff_global = 0.100000
nadd = 16
Loading 'hat3' ... done.
rescale = 1

    0 / 9
Segment   1/  1    1-6607
STEP 003-002-0  identical.    identical.    identical.    identical.    identical.   
Converged.

done
dvtditr (aa) Version 7.526
alg=A, model=BLOSUM62, 1.53, -0.00, -0.00, noshift, amax=0.0
0 thread(s)


Strate

## Nやgapの多い配列をalignmentから除去する

In [59]:
import pandas as pd
from Bio import SeqIO

In [60]:
def fasta_to_dataframe(fasta_file):
    ids = []
    sequences = []
    for record in SeqIO.parse(fasta_file, "fasta"):
        ids.append(record.id)
        sequences.append(str(record.seq))

    df = pd.DataFrame({
        "ID": ids,
        "seq": sequences
    })
    return df

In [107]:
seq = fasta_to_dataframe("al_FCoV2_comp_N_nt_prelim.filtered.fas")
seq

,ID,seq
0,OR295209.1,atggccacacagggacaacgcgtcaactggggagatgaaccttcca...
1,NC_002306.3,atggccacacagggacaacgcgtcaactggggagatgaaccttcca...
2,GQ152141.1,atggccacacagggacaacgcgtcaactggggagatgaaccttcca...
3,MT239439.1,atggccacacagggacaacgcgtcaactggggagatgaaccttcca...
4,JN634064.1,atggccacacagggacaacgcgtcaactggggagatgaaccttcca...
5,DQ286389.1,atggccacacagggacaacgcgtcaactggggagatgaaccttcca...
6,JQ408981.1,atggccacacagggacaacgcgtcaactggggagatgaaccttcca...


各配列の中で、’N’と”ー”がいくつ含まれるのかを数える

In [108]:
# count how many 'N' and '-' in each seq and add as a new column
seq["N_count"] = seq["seq"].str.count("N")
seq["gap_count"] = seq["seq"].str.count("-")
seq

,ID,seq,N_count,gap_count
0,OR295209.1,atggccacacagggacaacgcgtcaactggggagatgaaccttcca...,0,1
1,NC_002306.3,atggccacacagggacaacgcgtcaactggggagatgaaccttcca...,0,1
2,GQ152141.1,atggccacacagggacaacgcgtcaactggggagatgaaccttcca...,0,16
3,MT239439.1,atggccacacagggacaacgcgtcaactggggagatgaaccttcca...,0,1
4,JN634064.1,atggccacacagggacaacgcgtcaactggggagatgaaccttcca...,0,4
5,DQ286389.1,atggccacacagggacaacgcgtcaactggggagatgaaccttcca...,0,1
6,JQ408981.1,atggccacacagggacaacgcgtcaactggggagatgaaccttcca...,0,1


In [109]:
# アライメントの長さを取得
# length of seq alignment
length_of_alignment = seq["seq"].str.len().iloc[0] # 最後の.iloc[0]は、一番最初の行っていう意味。今回の場合は、アライメント後の配列で、全ての配列長は同じなので、どれか一つを取れば良い
print(f"Length of alignment: {length_of_alignment}")

Length of alignment: 1135


全体のアライメントの長さのうち、半分以上が'N'もしくは'-'の配列を除く。以下で行なっている処理は、'N'の数が配列長 * 0.2以下、かつ'-'の数が配列長 * 0.2以下の行のみを選択しています

In [110]:
# remove rows if they contain 'N' or '-' more than 50% of the length of alignment
seq_clean = seq[(seq["N_count"] <= length_of_alignment * 0.2) & (seq["gap_count"] <= length_of_alignment * 0.5)]
seq_clean

,ID,seq,N_count,gap_count
0,OR295209.1,atggccacacagggacaacgcgtcaactggggagatgaaccttcca...,0,1
1,NC_002306.3,atggccacacagggacaacgcgtcaactggggagatgaaccttcca...,0,1
2,GQ152141.1,atggccacacagggacaacgcgtcaactggggagatgaaccttcca...,0,16
3,MT239439.1,atggccacacagggacaacgcgtcaactggggagatgaaccttcca...,0,1
4,JN634064.1,atggccacacagggacaacgcgtcaactggggagatgaaccttcca...,0,4
5,DQ286389.1,atggccacacagggacaacgcgtcaactggggagatgaaccttcca...,0,1
6,JQ408981.1,atggccacacagggacaacgcgtcaactggggagatgaaccttcca...,0,1


ほとんどNみたいな配列を除いて、綺麗になったデータフレームから、IDとseqのカラムを使って、fasta形式で出力

In [111]:
# export as fasta
with open("al_FCoV2_comp_N_nt_prelim_cleaned.fas", "w") as f:
    for index, row in seq_clean.iterrows():
        f.write(f">{row['ID']}\n{row['seq']}\n")